# One-Port S11 Sweep -> .s1p + QCoDeS Database

Measures raw S11 on a chosen subset of the 6 MM4250 RF channels, saving each measurement both as a `.s1p` file and as a QCoDeS run in a shared database. Uses `scripts/oneport_db_sweep.py` (which in turn uses `scripts/oneport_sweep.py` for the measurement itself).

This is raw acquisition only -- no calibration or de-embedding is applied. Unlike `sweep_all_channels_1port`, you pick which channels to sweep rather than always doing all 6.

Every run accumulates into one database file, `mm4250_oneport.db` at this repo's root. Each call of the sweep below becomes its own QCoDeS *experiment*, named `<date_str>_<temp_str>_<switch_serials>`; each channel measured in that call becomes one *run* named `RF<n>` inside it. Browse the file afterwards with `plottr-inspectr --db mm4250_oneport.db`.

Before running: edit `channels`/`date_str`/`temp_str`/`switch_serials` in the config cell to match this run.

In [ ]:
import sys
from pathlib import Path

# drivers/ and scripts/ live right alongside this notebook in this repo
# -- make sure this repo's root is on sys.path regardless of Jupyter's
# working-directory behavior.
try:
    here = Path(_dh[0])  # Jupyter sets _dh[0] to this notebook's directory
except NameError:
    here = Path.cwd()
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250
from scripts.oneport_db_sweep import run_oneport_sweep

In [ ]:
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")

In [ ]:
# Edit these for each run:
channels = [1, 3, 5]         # any subset of 1-6; use list(range(1, 7)) for all of them
date_str = "YYYYMMDD"        # e.g. "20260917"
temp_str = "295K"            # e.g. "295K", "3K", "25mK"
switch_serials = "SN0001"    # the switch under test, e.g. "0030"

sweep_dir = run_oneport_sweep(channels, date_str, temp_str, switch_serials, vna=ksvna, switch=switch)
print(f"Sweep complete, .s1p files saved under {sweep_dir}")

In [ ]:
ksvna.close()
switch.close()